# PWM + DPU 통합 테스트 (모터 없이 핀 출력 검증)

**목적:** 새 `dpu.bit` (DPU+PWM+HW전처리/후처리 통합 RTL)를 로드하고 모터 없이 전체 파이프라인 정상 동작 확인

**신규 비트스트림 HW IP 구성:**
- `DPUCZDX8G_1` @ 0xB0000000 — DPU 추론 엔진
- `preprocess_top_0` — HW 전처리 (640×480→256×256 리사이즈, 300MHz)
- `postproc_top_0` @ 0x80010000 — HW 후처리 (마스크→조향오차 계산, 75MHz)
- `axi_dma_0` @ 0x80020000 — AXI DMA (PS DDR ↔ preprocess_top_0)
- `PWM_IP_{0-5}` @ 0xA000_0000~0xA005_0000 — 6채널 PWM (300MHz, 500Hz)

**테스트 순서:**
1. HWH 검증 (주소·클럭·채널 매핑이 default.yaml과 일치하는지)
2. dpu.bit 로드 + DPU 추론 검증
3. HW 전처리 (preprocess_top_0 + AXI DMA) 테스트
4. HW 후처리 (postproc_top_0) 레지스터 프로브
5. PWM 채널별 레지스터 직접 읽기/쓰기 검증
6. 제어 명령 → PWM 출력 변환 검증 (PynqMMIOActuator)
7. 전체 파이프라인 테스트 (test.mp4 → HW전처리 → DPU → HW후처리 → PWM)

---
## 1. 환경 설정 및 임포트

In [ ]:
import os
import sys
import time
import threading
from pathlib import Path

import cv2
import numpy as np
import yaml

def load_env_script(filepath="setup.sh"):
    if os.path.exists(filepath):
        with open(filepath) as f:
            for line in f:
                line = line.strip()
                if line.startswith("export "):
                    kv = line[7:].split("=", 1)
                    if len(kv) == 2:
                        os.environ[kv[0].strip()] = kv[1].strip()

load_env_script("setup.sh")

try:
    from pynq_dpu import DpuOverlay
    from pynq import MMIO
    BOARD = True
    print("[OK] pynq_dpu / pynq 임포트 성공 → 보드 모드")
except Exception as e:
    DpuOverlay = MMIO = None
    BOARD = False
    print(f"[FAIL] pynq 임포트 실패: {e}")
    raise SystemExit("보드 환경에서 실행하세요")

print(f"Python {sys.version}")
print(f"OpenCV {cv2.__version__}")

In [ ]:
CONFIG_DIR = "configs"

def load_yaml(path):
    with open(path) as f:
        return yaml.safe_load(f)

cfg = {
    "default": load_yaml(f"{CONFIG_DIR}/default.yaml"),
    "camera":  load_yaml(f"{CONFIG_DIR}/camera.yaml"),
    "model":   load_yaml(f"{CONFIG_DIR}/model.yaml"),
    "control": load_yaml(f"{CONFIG_DIR}/control.yaml"),
}

actuator_cfg = cfg["default"]["actuator"]
motor_cfg    = actuator_cfg["motors"]
OVERLAY_PATH = str(Path(actuator_cfg["overlay_path"]).resolve())

print(f"overlay: {OVERLAY_PATH}")
print(f"period_size: {actuator_cfg['period_size']}")
print(f"channels:")
for name, addr in motor_cfg["base_addrs"].items():
    print(f"  {name:<20} @ {addr}")

---
## 2. HWH 검증 + dpu.bit 로드 + DPU 초기화

In [ ]:
# ── HWH vs default.yaml 자동 검증 ─────────────────────────────────────────────
import xml.etree.ElementTree as ET

HWH_PATH = str(Path(actuator_cfg["overlay_path"]).with_suffix(".hwh").resolve())
print(f"HWH: {HWH_PATH}")

tree = ET.parse(HWH_PATH)
hwh_root = tree.getroot()

# clk_wiz 출력 클럭 맵
clk_map = {}
for mod in hwh_root.iter("MODULE"):
    if "clk_wiz" in mod.get("INSTANCE", "").lower():
        params = {p.get("NAME"): p.get("VALUE") for p in mod.iter("PARAMETER")}
        for i in range(1, 8):
            freq_key = f"C_CLKOUT{i}_OUT_FREQ"
            if freq_key in params:
                clk_map[f"clk_out{i}"] = float(params[freq_key])

print("\nclk_wiz_0 출력 클럭:")
for k, v in sorted(clk_map.items()):
    print(f"  {k}: {v:.0f} MHz")

# PWM IP 주소 및 AXI 클럭 확인
pwm_info = {}
for mr in hwh_root.iter("MEMRANGE"):
    inst = mr.get("INSTANCE", "")
    if "PWM_IP" in inst:
        pwm_info[inst] = {"addr": mr.get("BASEVALUE")}
for mod in hwh_root.iter("MODULE"):
    inst = mod.get("INSTANCE", "")
    if "PWM_IP" in inst:
        params = {p.get("NAME"): p.get("VALUE") for p in mod.iter("PARAMETER")}
        if inst in pwm_info:
            pwm_info[inst]["freq_hz"] = int(params.get("FREQ_HZ", 0))

print("\n=== PWM IP 검증 ===")
cfg_addrs = {name: int(addr, 0) for name, addr in motor_cfg["base_addrs"].items()}
pwm_by_addr = {int(v["addr"], 0): k for k, v in pwm_info.items()}

all_ok = True
for name, cfg_addr in cfg_addrs.items():
    hwh_ip = pwm_by_addr.get(cfg_addr, "NOT FOUND")
    match = hwh_ip != "NOT FOUND"
    all_ok = all_ok and match
    status = "✓" if match else "✗"
    print(f"  [{status}] {name:<20} @ {hex(cfg_addr)}  HWH: {hwh_ip}")

# PWM 클럭 주파수 확인
pwm_freq_hz = list(pwm_info.values())[0]["freq_hz"] if pwm_info else 0
pwm_clk_mhz = pwm_freq_hz / 1e6
expected_pwm_hz = pwm_freq_hz / actuator_cfg["period_size"]
print(f"\nPWM 클럭: {pwm_clk_mhz:.0f} MHz")
print(f"period_size: {actuator_cfg['period_size']}")
print(f"PWM 주파수: {expected_pwm_hz:.1f} Hz  (목표: 500 Hz)")

freq_ok = abs(expected_pwm_hz - 500) < 5
all_ok = all_ok and freq_ok
print(f"PWM 주파수 {'✓ 정상' if freq_ok else '✗ 불일치!'}")

# 신규 HW IP 확인
print("\n=== 신규 HW IP ===")
hw_ips = {"preprocess_top_0": None, "postproc_top_0": None, "axi_dma_0": None}
for mr in hwh_root.iter("MEMRANGE"):
    inst = mr.get("INSTANCE", "")
    if inst in hw_ips:
        hw_ips[inst] = mr.get("BASEVALUE")
for mod in hwh_root.iter("MODULE"):
    inst = mod.get("INSTANCE", "")
    if inst in hw_ips and hw_ips[inst] is None:
        params = {p.get("NAME"): p.get("VALUE") for p in mod.iter("PARAMETER")}
        hw_ips[inst] = params.get("C_BASEADDR", "N/A (streaming)")

# preprocess_top_0 파라미터
for mod in hwh_root.iter("MODULE"):
    if mod.get("INSTANCE") == "preprocess_top_0":
        p = {x.get("NAME"): x.get("VALUE") for x in mod.iter("PARAMETER")}
        hw_ips["preprocess_top_0"] = (
            f"N/A (AXI-Stream)  {p.get('IN_W')}x{p.get('IN_H')}→{p.get('OUT_W')}x{p.get('OUT_H')}"
            f"  {int(p.get('FREQ_HZ',0))//1000000}MHz"
        )
    if mod.get("INSTANCE") == "postproc_top_0":
        p = {x.get("NAME"): x.get("VALUE") for x in mod.iter("PARAMETER")}
        freq = int(p.get("FREQ_HZ", 0)) // 1000000
        hw_ips["postproc_top_0"] = f"0x80010000  {freq}MHz"
    if mod.get("INSTANCE") == "axi_dma_0":
        hw_ips["axi_dma_0"] = "0x80020000  MM2S+S2MM  300MHz"

for ip, info in hw_ips.items():
    print(f"  {ip:<22} : {info}")

HW_PREPROC_AVAIL = True
print(f"\n{'[OK] HWH 검증 통과' if all_ok else '[FAIL] HWH 불일치 — 비트스트림 또는 설정 확인 필요'}")


In [ ]:
# ── DPU Overlay 로드 ───────────────────────────────────────────────────────────
print(f"Loading overlay: {OVERLAY_PATH}")
t0 = time.time()
overlay = DpuOverlay(OVERLAY_PATH)
print(f"[OK] overlay 로드 완료 ({time.time()-t0:.1f}s)")

In [ ]:
# ── DPU Runner 초기화 + 서브그래프 분석 ──────────────────────────────────────
import xir
import vart

xmodel_path = Path(cfg["model"]["xmodel_path"])
assert xmodel_path.exists(), f"xmodel 없음: {xmodel_path}"

graph = xir.Graph.deserialize(str(xmodel_path))
root  = graph.get_root_subgraph()

# ── 서브그래프 분류 ────────────────────────────────────────────────────────────
# xmodel 파이프라인: USER(전처리/양자화) → DPU(추론) → CPU(역양자화)
DEVICE_ROLE = {
    "USER": "HW 전처리 대응 (float32→int8 양자화: input × IN_SCALE)",
    "DPU":  "DPU 하드웨어 추론",
    "CPU":  "HW 후처리 대응 (int8→float32 역양자화: output × OUT_SCALE)",
}

user_sgs, dpu_sg, cpu_sgs = [], None, []

print("=== xmodel 서브그래프 분석 ===")
for child in root.toposort_child_subgraph():
    dev = child.get_attr("device") if child.has_attr("device") else "UNKNOWN"
    role = DEVICE_ROLE.get(dev, dev)
    marker = " ← DPU" if dev == "DPU" else ""
    print(f"  [{dev}] {child.get_name()}{marker}")
    print(f"       역할: {role}")
    if dev == "DPU":
        dpu_sg = child
    elif dev == "USER":
        user_sgs.append(child)
    elif dev == "CPU":
        cpu_sgs.append(child)

assert dpu_sg is not None, "DPU subgraph 없음"
runner = vart.Runner.create_runner(dpu_sg, "run")

in_t   = runner.get_input_tensors()[0]
out_t  = runner.get_output_tensors()[0]
in_fp  = in_t.get_attr("fix_point")  if in_t.has_attr("fix_point")  else 0
out_fp = out_t.get_attr("fix_point") if out_t.has_attr("fix_point") else 0
IN_SCALE  = float(2 **  in_fp)   # USER 서브그래프 동작: float32 × IN_SCALE → int8
OUT_SCALE = float(2 ** -out_fp)  # CPU  서브그래프 동작: int8 × OUT_SCALE → float32

in_shape  = tuple(in_t.dims)    # (1, 256, 256, 3)
out_shape = tuple(out_t.dims)   # (1, 256, 256, 1)

in_buf  = np.empty(in_shape,  dtype=np.int8)
out_buf = [np.empty(out_shape, dtype=np.int8)]
out_f32 = np.empty(out_shape, dtype=np.float32)

print(f"\n[OK] DPU Runner 생성")
print(f"  입력: {in_shape}  IN_SCALE={IN_SCALE} (fix_point={in_fp})")
print(f"  출력: {out_shape}  OUT_SCALE={OUT_SCALE} (fix_point={out_fp})")
print(f"\nxmodel 전처리 (USER): float32 /255 × {IN_SCALE:.0f} → int8")
print(f"xmodel 후처리 (CPU) : int8 × {OUT_SCALE} → float32 로짓")
print(f"  threshold=0.0 적용 시: 로짓>0 ↔ sigmoid(로짓)>0.5 (등가)")

def dpu_infer(image_f32):
    """float32 RGB [0,1] → DPU 추론 → float32 로짓 출력
    USER 서브그래프(양자화)와 CPU 서브그래프(역양자화)를 수동 구현."""
    np.copyto(in_buf[0], (image_f32 * IN_SCALE).clip(-128, 127), casting="unsafe")
    jid = runner.execute_async([in_buf], out_buf)
    runner.wait(jid)
    np.multiply(out_buf[0], OUT_SCALE, out=out_f32)
    return out_f32


In [ ]:
# ── DPU 워밍업 + 추론 속도 측정 ────────────────────────────────────────────────
_, H, W, C = in_shape
dummy = np.random.rand(H, W, C).astype(np.float32)

for i in range(3):  # warmup
    dpu_infer(dummy)

times = []
for _ in range(10):
    t = time.time()
    out = dpu_infer(dummy)
    times.append((time.time() - t) * 1000)

print(f"[DPU 추론 속도] avg={np.mean(times):.1f}ms  min={np.min(times):.1f}ms  max={np.max(times):.1f}ms")
print(f"[DPU 출력 범위] min={out.min():.4f}  max={out.max():.4f}  shape={out.shape}")
print("[OK] DPU 추론 정상")

---
## 3. HW 전처리 테스트 (preprocess_top_0 + AXI DMA)

`preprocess_top_0`는 640×480 BGR 픽셀 스트림을 받아 256×256으로 리사이즈한 후
DPU에 맞는 int8 형식으로 출력하는 순수 AXI-Stream 하드웨어 IP입니다.

**데이터 경로:**
```
PS DDR (640×480×3 uint8)
  → AXI DMA MM2S  →  axis_dwidth_converter (32→24bit)
  → preprocess_top_0  (resize + BGR→RGB + 양자화)
  → axis_dwidth_converter (24→32bit)  →  AXI DMA S2MM
  → PS DDR (256×256×3 int8)
```

In [ ]:
# ── AXI DMA + preprocess_top_0 테스트 (직접 MMIO 방식 — 드라이버 혼용 hang 회피) ──
# 파이프라인: in(640x480 BGR uint8, 32bit packed [B,G,R,0])
#   → MM2S → preprocess_top_0 (nn_resize + bgr2rgb_normalize) → S2MM
#   → out(256x256, 32bit [R,G,B,0])
# ★ PYNQ dma.sendchannel 드라이버 + 수동리셋 혼용 시 .running 안풀려 hang →
#   직접 레지스터(RS/ADDR/LENGTH)로 전송하고 DMASR Idle 비트로 완료 판정.
from pynq import allocate

DMA_BASE = 0x80020000
MM2S_DMACR,MM2S_SR,MM2S_SA,MM2S_LEN = 0x00,0x04,0x18,0x28
S2MM_DMACR,S2MM_SR,S2MM_DA,S2MM_LEN = 0x30,0x34,0x48,0x58

HW_DMA_OK = True
IN_H, IN_W   = 480, 640
OUT_H, OUT_W = 256, 256
IN_BYTES  = IN_H  * IN_W  * 4   # 1228800
OUT_BYTES = OUT_H * OUT_W * 4   # 262144

in_dma_buf  = allocate(shape=(IN_BYTES,),  dtype=np.uint8)
out_dma_buf = allocate(shape=(OUT_BYTES,), dtype=np.int8)

cap_test = cv2.VideoCapture("test.mp4")
ret, test_frame = cap_test.read()
cap_test.release()
assert ret

test_frame_640 = cv2.resize(test_frame, (IN_W, IN_H))
_flat = test_frame_640.flatten()            # (921600,) BGR uint8
_frame_32 = np.zeros(IN_BYTES, dtype=np.uint8)
_frame_32[0::4] = _flat[0::3]  # B
_frame_32[1::4] = _flat[1::3]  # G
_frame_32[2::4] = _flat[2::3]  # R
in_dma_buf[:]  = _frame_32
in_dma_buf.flush()                          # CPU 캐시 -> DDR (MM2S가 읽기 전 필수)
out_dma_buf[:] = 0; out_dma_buf.flush()

print(f"입력  버퍼: {IN_BYTES:>7} bytes @ phys 0x{in_dma_buf.physical_address:08X}")
print(f"출력 버퍼: {OUT_BYTES:>7} bytes @ phys 0x{out_dma_buf.physical_address:08X}")

_dma = MMIO(DMA_BASE, 0x100)
# soft reset
_dma.write(MM2S_DMACR, 0x4)
while _dma.read(MM2S_DMACR) & 0x4: pass
_dma.write(S2MM_DMACR, 0x4)
while _dma.read(S2MM_DMACR) & 0x4: pass

t0 = time.time()
# S2MM 먼저 무장
_dma.write(S2MM_DMACR, 0x1)                 # RS=1
_dma.write(S2MM_DA,  out_dma_buf.physical_address)
_dma.write(S2MM_LEN, OUT_BYTES)             # 전송 시작
# MM2S 시작
_dma.write(MM2S_DMACR, 0x1)                 # RS=1
_dma.write(MM2S_SA,  in_dma_buf.physical_address)
_dma.write(MM2S_LEN, IN_BYTES)              # 전송 시작

DMA_TIMEOUT = 3.0
def _wait_idle(sr_off, name):
    while not (_dma.read(sr_off) & 0x2):     # Idle bit
        err = _dma.read(sr_off) & 0x70
        if err:
            raise RuntimeError(f"{name} DMA err=0x{err:02X} SR=0x{_dma.read(sr_off):08X}")
        if time.time() - t0 > DMA_TIMEOUT:
            raise RuntimeError(f"{name} DMA timeout {DMA_TIMEOUT}s SR=0x{_dma.read(sr_off):08X}")
        time.sleep(0.002)

try:
    _wait_idle(MM2S_SR, "MM2S")
    _wait_idle(S2MM_SR, "S2MM")
except RuntimeError as e:
    HW_DMA_OK = False; hw_out_int8 = None
    print(f"\n[FAIL] {e}")
    del in_dma_buf, out_dma_buf
    raise

hw_preproc_ms = (time.time() - t0) * 1000
out_dma_buf.invalidate()                    # DDR -> CPU 캐시 동기화 (필수)

mm2s_sr = _dma.read(MM2S_SR); s2mm_sr = _dma.read(S2MM_SR)
print(f"\n DMA 진단:")
print(f"  MM2S SR=0x{mm2s_sr:08X}  err={'OK' if not (mm2s_sr&0x70) else hex(mm2s_sr&0x70)}  Idle={(mm2s_sr>>1)&1}")
print(f"  S2MM SR=0x{s2mm_sr:08X}  err={'OK' if not (s2mm_sr&0x70) else hex(s2mm_sr&0x70)}  Idle={(s2mm_sr>>1)&1}")

# 32bit 출력 파싱: [R,G,B,0] -> [R,G,B]
hw_out_int8 = out_dma_buf.reshape(OUT_H, OUT_W, 4)[:, :, :3].copy()
print(f"\n[OK] HW 전처리 완료 ({hw_preproc_ms:.1f}ms)")
print(f"  출력 shape : {hw_out_int8.shape}  dtype={hw_out_int8.dtype}")
print(f"  값 범위    : min={hw_out_int8.min()}  max={hw_out_int8.max()}  nonzero={int(np.count_nonzero(hw_out_int8))}")

cpu_rsz  = cv2.resize(test_frame_640, (OUT_W, OUT_H), interpolation=cv2.INTER_NEAREST)
cpu_rgb  = cv2.cvtColor(cpu_rsz, cv2.COLOR_BGR2RGB)
cpu_q    = (cpu_rgb.astype(np.float32) / 255.0 * IN_SCALE).clip(-128,127).astype(np.int8)
max_diff = int(np.abs(hw_out_int8.astype(np.int16) - cpu_q.astype(np.int16)).max())
mean_diff = float(np.abs(hw_out_int8.astype(np.int16) - cpu_q.astype(np.int16)).mean())
print(f"  CPU(nearest) 대비 최대 오차: {max_diff}  평균 오차: {mean_diff:.2f}")
print(f"  (nearest-neighbor 경계 픽셀 차이로 일부 큰 오차 가능, 평균이 작으면 정상)")

del in_dma_buf, out_dma_buf
print("[OK] preprocess_top_0 테스트 완료")


---
## 4. HW 후처리 프로브 (postproc_top_0)

`postproc_top_0`는 DPU 출력 마스크(256×256×1 int8)를 DDR에서 직접 읽어
차선 중심선과 조향 오차를 계산하는 하드웨어 IP입니다.

**RTL 확인된 레지스터 맵 (ADDR_WIDTH=6):**
- `0x00` CTRL/STATUS: W bit0=start / R {29'd0, done_latch, busy, 1'b0}
- `0x04` ADDR_LO: ddr_addr[31:0]
- `0x08` RESULT_ERR: steering_error Q1.15 (÷32768 → float)
- `0x0C` RESULT_VALID: bit0
- `0x10` RESULT_PIXELS: lane_pixels[15:0]
- `0x18` ADDR_HI: ddr_addr[39:32]

> **섹션 3이 DMA 타임아웃/중단됐으면 섹션 2 (overlay 로드)를 먼저 재실행하세요.**  
> AXI 버스가 잠긴 상태에서 이 셀을 실행하면 `PP.read()`에서 hang합니다.

In [ ]:
# ── postproc_top_0 레지스터 맵 (RTL 확인 완료) ───────────────────────────────
#  0x00  CTRL/STATUS  W:bit0=start  R:{done_latch[2], busy[1], 1'b0}
#  0x04  ADDR_LO      ddr_addr[31:0]
#  0x08  RESULT_ERR   (center_x-128)*256  signed Q1.15  → /32768.0
#  0x0C  RESULT_VALID bit0
#  0x10  RESULT_PIXELS lane_pixels[15:0]
#  0x18  ADDR_HI      ddr_addr[39:32]
#  done_latch: 0x00 읽기 시 자동 클리어
# ─────────────────────────────────────────────────────────────────────────────
# ★ 이 셀 실행 전 섹션 3이 DMA hang으로 중단됐다면:
#   섹션 2 (overlay 로드) 재실행 → 이 셀 바로 실행  (섹션 3 건너뛰어도 됨)
# ─────────────────────────────────────────────────────────────────────────────

POSTPROC_BASE  = 0x80010000
POSTPROC_RANGE = 0x1000
PP = MMIO(POSTPROC_BASE, POSTPROC_RANGE)

PP_CTRL        = 0x00
PP_ADDR_LO     = 0x04
PP_RESULT_ERR  = 0x08
PP_RESULT_VLD  = 0x0C
PP_RESULT_PX   = 0x10
PP_ADDR_HI     = 0x18

print("=== postproc_top_0 초기 레지스터 상태 ===")
for off, name in [(PP_CTRL,"CTRL/STATUS"),(PP_ADDR_LO,"ADDR_LO"),
                  (PP_RESULT_ERR,"RESULT_ERR"),(PP_RESULT_VLD,"RESULT_VALID"),
                  (PP_RESULT_PX,"RESULT_PIXELS"),(PP_ADDR_HI,"ADDR_HI")]:
    v = PP.read(off)
    print(f"  0x{off:02X} {name:<15} = 0x{v:08X}")

print("\n=== postproc_top_0 기동 테스트 ===")

cap_tmp = cv2.VideoCapture("test.mp4")
ret, frm = cap_tmp.read(); cap_tmp.release()
frm = cv2.resize(frm, (256, 256))
frm = cv2.cvtColor(frm, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
dpu_infer(frm)  # out_buf[0] 채움

from pynq import allocate as _alloc
dpu_out_cma = _alloc(shape=out_shape, dtype=np.int8)
np.copyto(dpu_out_cma, out_buf[0])
phys = dpu_out_cma.physical_address
print(f"  DPU 출력 CMA @ 0x{phys:010X}")

PP.write(PP_ADDR_LO, phys & 0xFFFFFFFF)
PP.write(PP_ADDR_HI, (phys >> 32) & 0xFF)

st = PP.read(PP_CTRL)
busy_now = (st >> 1) & 1
print(f"  기동 전  STATUS=0x{st:08X}  busy={busy_now}")
if busy_now:
    for _ in range(50):
        time.sleep(0.01)
        if not ((PP.read(PP_CTRL) >> 1) & 1): break

PP.write(PP_CTRL, 0x1)
t0 = time.time()

for _ in range(400):
    time.sleep(0.001)
    st = PP.read(PP_CTRL)
    if (st >> 2) & 1: break  # done_latch

elapsed = (time.time() - t0) * 1000
done_bit = (st >> 2) & 1
busy_bit = (st >> 1) & 1
print(f"  완료  {elapsed:.1f}ms  STATUS=0x{st:08X}  done={done_bit} busy={busy_bit}")

if not done_bit:
    if busy_bit:
        print("\n[WARN] done_latch 미세트 (busy=1) — postproc AXI Master가 DDR 읽기 hang")
        print("       섹션 2 (overlay 로드) 재실행 필요")
    else:
        print("\n[WARN] done_latch 미세트 (busy=0) — start 펄스가 FSM에 도달하지 못함")

raw = PP.read(PP_RESULT_ERR)
vld = PP.read(PP_RESULT_VLD) & 1
px  = PP.read(PP_RESULT_PX) & 0xFFFF
i16 = raw if raw < 0x80000000 else raw - 0x100000000
steer = i16 / 32768.0

print(f"\n  RESULT_ERR    = 0x{raw:08X}  i16={i16:+6d}  → steer={steer:+.4f}")
print(f"  RESULT_VALID  = {vld}")
print(f"  RESULT_PIXELS = {px}")

if vld and -1.0 <= steer <= 1.0:
    print("\n[OK] postproc_top_0 정상 동작 확인")
    try:
        steer_py, lane_py = postprocess(out_f32, 256, 256, 0.0)
        if steer_py is not None:
            diff = abs(steer - steer_py)
            print(f"     HW steer={steer:+.4f}  CPU steer={steer_py:+.4f}  차이={diff:.4f}")
            print(f"     (RTL 정수 나눗셈 절삭 vs CPU float → 최대 ±1/128≈0.0078)")
        else:
            print(f"     HW steer={steer:+.4f}  (CPU: 차선 없음)")
    except NameError:
        print(f"     HW steer={steer:+.4f}  px={px}")
        print(f"     (CPU 비교 생략 — 섹션 7 헬퍼 셀 먼저 실행하면 비교 가능)")
elif not vld:
    print("\n[INFO] result_valid=0 — 유효 차선 없음")

del dpu_out_cma


---
## 5. PWM 채널별 레지스터 직접 검증

모터 없이 각 채널의 MMIO 레지스터(PERIOD / DUTY / VALID)를 직접 쓰고 읽어서
PWM IP가 정상적으로 응답하는지 확인합니다.

In [ ]:
# ── 안전 체크: overlay 로드 여부 확인 ──────────────────────────────────────────
try:
    overlay
    print(f"[OK] overlay 확인: {OVERLAY_PATH}")
except NameError:
    raise RuntimeError(
        "[FAIL] overlay 미로드!\n"
        "섹션 2 (dpu.bit 로드) 셀을 먼저 실행한 뒤 이 셀을 실행하세요."
    )

# clk_wiz_0 clk_out2 = 300MHz 안정화 대기
time.sleep(1.0)

# ── MMIO 인스턴스 생성 ─────────────────────────────────────────────────────────
REG_PERIOD = 0x00
REG_DUTY   = 0x04
REG_VALID  = 0x08

# PWM IP 클럭: clk_wiz_0 clk_out2 = 300MHz
# 300MHz / 600600 = 499.5Hz ≈ 500Hz PWM
PERIOD     = int(actuator_cfg["period_size"])
MMIO_RANGE = int(actuator_cfg.get("mmio_range", 0x10000))

def parse_addr(addr):
    return int(addr, 0) if isinstance(addr, str) else int(addr)

mmios = {
    name: MMIO(parse_addr(addr), MMIO_RANGE)
    for name, addr in motor_cfg["base_addrs"].items()
}

for name, m in mmios.items():
    m.write(REG_PERIOD, PERIOD)
    m.write(REG_DUTY,   PERIOD)
    m.write(REG_VALID,  0)

print(f"MMIO 초기화 완료 ({len(mmios)}개 채널)")
print(f"  클럭: clk_out2 = 300MHz")
print(f"  PERIOD = {PERIOD}  →  PWM ≈ {300_000_000/PERIOD:.1f} Hz")


In [ ]:
# ── 채널별 레지스터 읽기/쓰기 테스트 ─────────────────────────────────────────
def test_channel(name, mmio, test_duty_pct=0.5):
    """단일 채널에 duty 값을 쓰고 읽어서 일치 여부 반환"""
    duty = int(PERIOD * test_duty_pct)

    mmio.write(REG_PERIOD, PERIOD)
    mmio.write(REG_DUTY,   duty)
    mmio.write(REG_VALID,  1)
    time.sleep(0.01)

    r_period = mmio.read(REG_PERIOD)
    r_duty   = mmio.read(REG_DUTY)
    r_valid  = mmio.read(REG_VALID)

    ok_period = (r_period == PERIOD)
    ok_duty   = (r_duty   == duty)
    ok_valid  = (r_valid  == 1)
    passed    = ok_period and ok_duty and ok_valid

    status = "PASS" if passed else "FAIL"
    print(
        f"  [{status}] {name:<20}  "
        f"PERIOD={r_period}({'OK' if ok_period else f'expect {PERIOD}'})  "
        f"DUTY={r_duty}({'OK' if ok_duty else f'expect {duty}'})  "
        f"VALID={r_valid}({'OK' if ok_valid else 'expect 1'})"
    )

    # 테스트 후 VALID=0 (꺼두기)
    mmio.write(REG_VALID, 0)
    return passed


print("=== PWM 채널 레지스터 테스트 (duty=50%) ===")
results = {name: test_channel(name, m) for name, m in mmios.items()}

passed = sum(results.values())
total  = len(results)
print(f"\n결과: {passed}/{total} 채널 정상")
if passed == total:
    print("[OK] 모든 PWM 채널 레지스터 읽기/쓰기 정상")
else:
    failed = [n for n, ok in results.items() if not ok]
    print(f"[FAIL] 문제 채널: {failed}")

In [ ]:
# ── 채널별 duty 스윕 테스트 (10% ~ 90%) ──────────────────────────────────────
print("=== PWM Duty 스윕 테스트 ===")
for pct in [0.1, 0.25, 0.5, 0.75, 0.9]:
    duty = int(PERIOD * pct)
    errors = []
    for name, m in mmios.items():
        m.write(REG_DUTY, duty)
        m.write(REG_VALID, 1)
        r = m.read(REG_DUTY)
        if r != duty:
            errors.append(f"{name}(got {r})")
        m.write(REG_VALID, 0)
    status = "PASS" if not errors else f"FAIL: {errors}"
    print(f"  duty={pct*100:4.0f}%  ({duty:>6} clk)  → [{status}]")

print("[OK] Duty 스윕 테스트 완료")

---
## 6. 제어 명령 → PWM 출력 변환 검증 (PynqMMIOActuator)

steering_cmd / speed_cmd 값에 따라 올바른 채널이 활성화되고
레지스터 값이 기대값과 일치하는지 확인합니다.

※ `steering_center_hold_percent=0.3`이므로 steer=0일 때 양쪽 조향 채널이 center hold duty로 동시 활성화됩니다.

In [ ]:
# ── PynqMMIOActuator 클래스 (원본에서 그대로 가져옴) ──────────────────────────
def _clamp(v, lo, hi):
    return max(lo, min(hi, v))

class PynqMMIOActuator:
    def __init__(self, cfg=None):
        self.cfg              = cfg or {}
        self.mmios            = {}
        self.initialized      = False
        self._last_cmd_time   = 0.0
        self._watchdog_stop   = threading.Event()
        self._watchdog_thread = None

    @staticmethod
    def _parse_addr(addr):
        return int(addr, 0) if isinstance(addr, str) else int(addr)

    def initialize(self):
        motor_cfg   = self.cfg.get("motors", {})
        base_addrs  = motor_cfg.get("base_addrs", {})
        mmio_range  = int(self.cfg.get("mmio_range", 0x10000))
        self.mmios  = {name: MMIO(self._parse_addr(addr), mmio_range)
                       for name, addr in base_addrs.items()}
        self._period             = int(self.cfg.get("period_size", 200000))
        self._drive_duty_pct     = float(self.cfg.get("drive_duty_percent", 1.0))
        self._steer_duty_pct     = float(self.cfg.get("steering_duty_percent", 1.0))
        self._steer_min_duty_pct = float(self.cfg.get("steering_min_duty_percent", 0.0))
        center_pct               = float(self.cfg.get("steering_center_hold_percent", 0.0))
        self._steer_center_duty  = int(self._period * self._steer_duty_pct * center_pct)
        self._fwd_channels       = motor_cfg.get("drive_channels", [])
        self._bwd_channels       = motor_cfg.get("reverse_channels", [])
        self._r_name             = motor_cfg.get("steering_right")
        self._l_name             = motor_cfg.get("steering_left")
        self._steer_deadband     = float(self.cfg.get("steering_feedback_deadband", 0.05))
        for name in self.mmios:
            self.mmios[name].write(REG_PERIOD, self._period)
            self.mmios[name].write(REG_DUTY,   self._period)
            self.mmios[name].write(REG_VALID,  0)
        self.initialized = True
        self._last_cmd_time = time.time()
        self._watchdog_stop.clear()
        self._watchdog_thread = threading.Thread(target=self._watchdog_loop, daemon=True)
        self._watchdog_thread.start()
        print(f"[MMIO] 초기화 완료  period={self._period}  "
              f"drive_duty={self._drive_duty_pct:.0%}  "
              f"steer_duty={self._steer_duty_pct:.0%}  "
              f"steer_min={self._steer_min_duty_pct:.0%}")

    def _watchdog_loop(self):
        timeout = float(self.cfg.get("command_timeout_sec", 0.3))
        while not self._watchdog_stop.is_set():
            self._watchdog_stop.wait(timeout=0.2)
            if self._watchdog_stop.is_set():
                break
            if timeout > 0 and (time.time() - self._last_cmd_time) > timeout:
                print("[Watchdog] 타임아웃 → 정지")
                self.stop()
                break

    def _write_duty(self, name, value):
        if name in self.mmios:
            self.mmios[name].write(REG_DUTY, int(value))

    def _write_valid(self, name, enable):
        if name in self.mmios:
            if enable:
                self.mmios[name].write(REG_PERIOD, self._period)
            self.mmios[name].write(REG_VALID, 1 if enable else 0)

    def _steer_duty(self, effort_abs):
        min_pct   = self._steer_min_duty_pct
        effective = min_pct + (1.0 - min_pct) * _clamp(effort_abs, 0.0, 1.0)
        return int(self._period * self._steer_duty_pct * effective)

    def _read_all_regs(self):
        return {
            name: {
                "period": int(m.read(REG_PERIOD)),
                "duty":   int(m.read(REG_DUTY)),
                "valid":  int(m.read(REG_VALID)),
            }
            for name, m in self.mmios.items()
        }

    def apply_control(self, steering_cmd, speed_cmd):
        steer = _clamp(float(steering_cmd), -1.0, 1.0)
        speed = _clamp(float(speed_cmd),    -1.0, 1.0)

        # 전진/후진
        for name in self._fwd_channels + self._bwd_channels:
            self._write_valid(name, False)
        if abs(speed) >= 0.05:
            drive_duty = int(self._period * self._drive_duty_pct * abs(speed))
            channels   = self._fwd_channels if speed > 0 else self._bwd_channels
            for name in channels:
                self._write_duty(name, drive_duty)
                self._write_valid(name, True)

        # 조향
        self._write_valid(self._r_name, False)
        self._write_valid(self._l_name, False)
        if abs(steer) < self._steer_deadband:
            if self._steer_center_duty > 0:
                self._write_duty(self._r_name, self._steer_center_duty)
                self._write_duty(self._l_name, self._steer_center_duty)
                self._write_valid(self._r_name, True)
                self._write_valid(self._l_name, True)
        else:
            duty = self._steer_duty(abs(steer))
            name = self._r_name if steer > 0 else self._l_name
            self._write_duty(name, duty)
            self._write_valid(name, True)

        self._last_cmd_time = time.time()
        return self._read_all_regs()

    def stop(self):
        for name in self.mmios:
            self._write_valid(name, False)

    def close(self):
        self._watchdog_stop.set()
        self.stop()
        self.initialized = False


actuator = PynqMMIOActuator(actuator_cfg)
actuator.initialize()

In [ ]:
# ── 제어 명령 → PWM 레지스터 검증 ─────────────────────────────────────────────
# steering_center_hold_percent=0.3 이므로 steer=0일 때 양쪽 조향 채널이 동시에
# center_hold duty로 활성화됩니다 (기구적 중앙 복귀력 제공 의도).
test_cases = [
    # (설명,           steering_cmd, speed_cmd,  기대 활성 채널)
    ("직진",            0.0,  0.25, ["rear_right_fwd", "rear_left_fwd",
                                     "steering_right", "steering_left"]),  # center hold 활성
    ("우회전",          0.8,  0.25, ["rear_right_fwd", "rear_left_fwd", "steering_right"]),
    ("좌회전",         -0.8,  0.25, ["rear_right_fwd", "rear_left_fwd", "steering_left"]),
    ("후진",            0.0, -0.25, ["rear_right_bwd", "rear_left_bwd",
                                     "steering_right", "steering_left"]),  # center hold 활성
    ("정지+center hold",0.0,  0.0,  ["steering_right", "steering_left"]), # center hold만
]

print("=== 제어 명령 → PWM 출력 검증 ===")
print("(steering_center_hold_percent=0.3: steer=0 시 양쪽 조향 채널 center hold 활성)")
all_pass = True
for desc, steer, speed, expected_active in test_cases:
    regs = actuator.apply_control(steer, speed)
    time.sleep(0.02)

    active_set   = set(n for n, r in regs.items() if r["valid"] == 1)
    expected_set = set(expected_active)
    passed = (active_set == expected_set)
    if not passed:
        all_pass = False

    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {desc:<18} steer={steer:+.2f} speed={speed:+.2f}")
    if not passed:
        print(f"          기대={sorted(expected_set)}")
        print(f"          실제={sorted(active_set)}")
    for ch in sorted(active_set):
        r = regs[ch]
        pct = r['duty'] / r['period'] * 100 if r['period'] > 0 else 0
        print(f"          └ {ch:<20}  duty={r['duty']:>6} ({pct:.1f}%)  valid={r['valid']}")

actuator.stop()
print(f"\n결과: {'[OK] 모든 케이스 통과' if all_pass else '[FAIL] 일부 케이스 실패'}")


In [ ]:
# ── steering_cmd 연속 스윕 → duty 선형성 확인 ─────────────────────────────────
print("=== 조향 Duty 선형성 테스트 ===")
print(f"  {'steering_cmd':>14}  {'채널':>20}  {'duty':>8}  {'%':>6}")
print("  " + "-" * 54)

for steer in [-1.0, -0.8, -0.5, -0.3, 0.3, 0.5, 0.8, 1.0]:
    regs = actuator.apply_control(steer, 0.0)
    time.sleep(0.01)
    for name, r in regs.items():
        if r["valid"] == 1 and ("steering" in name):
            pct = r["duty"] / r["period"] * 100 if r["period"] > 0 else 0
            print(f"  {steer:>+14.2f}  {name:>20}  {r['duty']:>8}  {pct:>5.1f}%")

actuator.stop()
print("[OK] 조향 선형성 테스트 완료")

---
## 8. 정리

In [ ]:
try:
    actuator.close()
except: pass
try:
    actuator_re.close()
except: pass
print("[OK] 자원 해제 완료")
print()
print("=== 테스트 요약 ===")
print("1. HWH 검증                      → 섹션 2-0")
print("2. dpu.bit 로드 + DPU 추론       → 섹션 2-1~3")
print("3. HW 전처리 (DMA)               → 섹션 3 (개별 IP 검증용)")
print("4. HW 후처리 레지스터 프로브     → 섹션 4")
print("5. PWM 레지스터 읽기/쓰기 검증   → 섹션 5")
print("6. 제어 명령 → PWM 출력 검증     → 섹션 6")
print("후처리 성능/정합성 비교는 프로젝트 루트의 compare_postprocessing.py 사용")
